# OneDrive to Google Drive Transfer (Token-Based)
# OneDrive到Google Drive传输（基于Token）

**No admin permissions needed - just paste access tokens**

**无需管理员权限 - 只需粘贴访问令牌**

---

## Important Notes:

- Transfer time: **2-4 hours** for 481GB
- Token expires every **1 hour**
- You'll need to update token **2-3 times** during transfer
- Keep this tab open and check back every 50 minutes

---

## Step 1: Install Rclone
## 步骤1：安装Rclone

In [ ]:
# Install rclone
!curl https://rclone.org/install.sh | sudo bash

# Verify installation
!rclone version

print("\n✅ Rclone installed successfully!")

## Step 2: Mount Google Drive
## 步骤2：挂载Google Drive

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted!")
print("📁 Destination: /content/drive/MyDrive/Egolife_videos")

## Step 3: Configure OneDrive with Access Token
## 步骤3：用访问令牌配置OneDrive

**Get your access token:**
1. Go to: https://developer.microsoft.com/en-us/graph/graph-explorer
2. Sign in with UNC account (zengqi@AD.UNC.EDU)
3. Click "Access token" tab
4. Copy the entire token
5. Paste below

In [ ]:
import json
import time

print("="*70)
print("PASTE YOUR UNC ONEDRIVE ACCESS TOKEN")
print("粘贴你的UNC OneDrive访问令牌")
print("="*70)
print()

# Get token from user
access_token = input("Paste access token here: ").strip()

if not access_token:
    raise ValueError("❌ No token provided!")

# Create rclone config with access token
config = {
    "unc_onedrive": {
        "type": "onedrive",
        "token": json.dumps({
            "access_token": access_token,
            "token_type": "Bearer",
            "expiry": time.strftime("%Y-%m-%dT%H:%M:%S.000000000Z", 
                                   time.gmtime(time.time() + 3600))
        }),
        "drive_type": "business"
    }
}

# Write config file
with open('/root/.config/rclone/rclone.conf', 'w') as f:
    for remote_name, remote_config in config.items():
        f.write(f"[{remote_name}]\n")
        for key, value in remote_config.items():
            f.write(f"{key} = {value}\n")
        f.write("\n")

print("\n✅ OneDrive configured!")
print("\n📋 Testing connection...")

# Test connection
!rclone lsd unc_onedrive: --max-depth 1

print("\n✅ Connection successful!")

## Step 4: Verify Source Files
## 步骤4：验证源文件

In [ ]:
print("📂 Checking Egolife_videos folder...\n")

# List first 10 files
!rclone ls unc_onedrive:Egolife_videos --max-depth 1 | head -n 10

# Count total files
print("\n📊 Counting files...")
!rclone ls unc_onedrive:Egolife_videos | wc -l
print("Expected: 240 files")

# Check total size
print("\n📊 Total size:")
!rclone size unc_onedrive:Egolife_videos

## Step 5: Create Destination Folder
## 步骤5：创建目标文件夹

In [ ]:
!mkdir -p "/content/drive/MyDrive/Egolife_videos"

print("✅ Destination folder created!")

## Step 6: START TRANSFER
## 步骤6：开始传输

**IMPORTANT:**
- This will take 2-4 hours
- Your token will expire in 1 hour
- When you see "401 Unauthorized" error, come back and run Step 7
- Don't close this tab!

**重要：**
- 这将需要2-4小时
- 你的令牌将在1小时后过期
- 当你看到"401 Unauthorized"错误时，回来运行步骤7
- 不要关闭这个标签页！

In [ ]:
from datetime import datetime

print("🚀 Starting transfer...")
print(f"⏰ Start time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("\n" + "="*70)
print("TRANSFERRING 481GB (240 videos)")
print("Expected time: 2-4 hours")
print("Token will expire in ~1 hour - watch for errors!")
print("="*70 + "\n")

# Start transfer
!rclone copy \
  unc_onedrive:Egolife_videos \
  "/content/drive/MyDrive/Egolife_videos" \
  --progress \
  --transfers 8 \
  --checkers 8 \
  --buffer-size 128M \
  --drive-chunk-size 64M \
  --stats 30s \
  --stats-one-line \
  --log-file /content/transfer.log \
  --ignore-errors \
  -v

print("\n" + "="*70)
print(f"⏰ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("="*70)
print("\nIf you see 401 errors above, run Step 7 to update token.")
print("如果上面看到401错误，运行步骤7更新令牌。")

## Step 7: UPDATE TOKEN (Run when token expires)
## 步骤7：更新令牌（令牌过期时运行）

**When to run this:**
- When you see "401 Unauthorized" errors in Step 6
- Or every 50 minutes to be safe

**何时运行：**
- 当你在步骤6看到"401 Unauthorized"错误时
- 或者每50分钟运行一次以确保安全

In [ ]:
import json
import time

print("="*70)
print("UPDATE ACCESS TOKEN")
print("更新访问令牌")
print("="*70)
print()
print("Get new token from: https://developer.microsoft.com/en-us/graph/graph-explorer")
print()

# Get new token
new_token = input("Paste NEW access token here: ").strip()

if not new_token:
    raise ValueError("❌ No token provided!")

# Update config
config = {
    "unc_onedrive": {
        "type": "onedrive",
        "token": json.dumps({
            "access_token": new_token,
            "token_type": "Bearer",
            "expiry": time.strftime("%Y-%m-%dT%H:%M:%S.000000000Z", 
                                   time.gmtime(time.time() + 3600))
        }),
        "drive_type": "business"
    }
}

# Write updated config
with open('/root/.config/rclone/rclone.conf', 'w') as f:
    for remote_name, remote_config in config.items():
        f.write(f"[{remote_name}]\n")
        for key, value in remote_config.items():
            f.write(f"{key} = {value}\n")
        f.write("\n")

print("\n✅ Token updated!")
print(f"⏰ Time: {time.strftime('%Y-%m-%d %H:%M:%S')}")
print("\n🔄 Now RE-RUN Step 6 to continue transfer!")
print("🔄 现在重新运行步骤6以继续传输！")

## Step 8: Verify Transfer
## 步骤8：验证传输

**Run this AFTER transfer completes (no more errors in Step 6)**

In [ ]:
print("📊 Verifying transfer...\n")

# Count files
file_count = !ls "/content/drive/MyDrive/Egolife_videos" | wc -l
print(f"✅ Files transferred: {file_count[0]}")
print("   Expected: 240 files\n")

# Check total size
!du -sh "/content/drive/MyDrive/Egolife_videos"

# List first 10 files
print("\n📁 First 10 files:")
!ls "/content/drive/MyDrive/Egolife_videos" | head -n 10

if int(file_count[0]) == 240:
    print("\n" + "="*70)
    print("✅ ✅ ✅ TRANSFER SUCCESSFUL! ✅ ✅ ✅")
    print("All 240 videos transferred to Google Drive!")
    print("="*70)
else:
    print(f"\n⚠️  Found {file_count[0]} files (expected 240)")
    print("Check the transfer log for errors.")

## Step 9: Check Transfer Log (If Needed)
## 步骤9：查看传输日志（如需要）

In [ ]:
# Show last 50 lines of log
!tail -n 50 /content/transfer.log

## ✅ Complete!

**Next:** Contact me to generate public URLs for all videos.

---

## Summary of Steps:

1. Run Steps 1-6 once
2. Every ~50 minutes:
   - Get new token from Graph Explorer
   - Run Step 7 to update token
   - Re-run Step 6 to continue transfer
3. When Step 6 finishes with no errors, run Step 8 to verify

**Total time:** 2-4 hours (you'll update token 2-3 times)